In [96]:
from pyspark.sql import SparkSession

In [97]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, FloatType, DoubleType, DecimalType, DateType
from pyspark.sql.window import Window

In [98]:
spark = (
    SparkSession.builder
        .appName("minio_app_ minio")
        .master("spark://spark-master:7077")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.access.key", "admin")
        .config("spark.hadoop.fs.s3a.secret.key", "admin123")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.sql.extensions","io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog","org.apache.spark.sql.delta.catalog.DeltaCatalog")

        .getOrCreate()
)

In [99]:
# Caminho da Silver
silver_path = "s3a://analytics/medallion/02-silver/vendas"

In [100]:

# Ler o Delta Silver
df_silver = spark.read.format("delta").load(silver_path)


In [101]:
df_silver.show(5)

+----------+--------------+--------------+----------------+--------------+------------+--------------------+----------+--------------+-----------+--------------------+
|data_venda|  cliente_nome|          Pais|      continente|     categoria|       marca|             produto|quantidade|preco_unitario|total_venda|          updated_at|
+----------+--------------+--------------+----------------+--------------+------------+--------------------+----------+--------------+-----------+--------------------+
|2018-09-10| Isaiah Wright|        Brasil|  América do Sul|Sistema de Som|     Litware|Sistema de Som 5....|         1|        579.00|     579.00|2026-03-11 17:44:...|
|2018-09-10|  Sierra Young|Estados Unidos|América do Norte|Sistema de Som|     Litware|Sistema de Som 5....|         3|        579.00|    1737.00|2026-03-11 17:44:...|
|2018-09-10| Cameron Lewis|Estados Unidos|América do Norte|Sistema de Som|     Litware|Sistema de Som 5....|         2|        579.00|    1158.00|2026-03-11 17:

In [102]:
df_silver.select("Pais").distinct().show()

+-------------+
|         Pais|
+-------------+
|       Suécia|
|       Canadá|
|      Polônia|
|       França|
|        Suíça|
|     Alemanha|
|        Butão|
|      Armênia|
|       Rússia|
|    Austrália|
|        China|
|Coreia do Sul|
|      Romênia|
|        Malta|
|        Chile|
|    Singapura|
|        Taiwã|
|    Tailândia|
|      Espanha|
|    Eslovênia|
+-------------+
only showing top 20 rows



df_silver.show(5)

In [103]:
gold_path_cliente = "s3a://analytics/medallion/03-gold/dim_cliente" 
gold_path_produto = "s3a://analytics/medallion/03-gold/dim_produto"
gold_path_regiao = "s3a://analytics/medallion/03-gold/dim_regiao"
gold_path_tempo = "s3a://analytics/medallion/03-gold/dim_tempo"
gold_path_fato = "s3a://analytics/medallion/03-gold/fato_vendas"

In [104]:
# Criar Dimensão Cliente a partir da Silver

# janela ordenada
window_esp = Window.orderBy("nome_completo")


# Criar Dimensão Cliente
dim_cliente = (
    df_silver
        .select(F.col("cliente_nome").alias("nome_completo"))
        .distinct()
        .withColumn(
            "id",
            F.row_number().over(window_esp)
        )
        .select("id", "nome_completo")
)

(
    dim_cliente.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(gold_path_cliente)
)


26/03/12 01:11:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 01:11:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 01:11:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 01:11:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 01:11:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 01:11:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 0

In [105]:

# Criar Dimensão Produto a partir da Silver

# janela ordenada
window_esp = Window.orderBy("produto")

# Criar Dimensão Produto
dim_produto = (
    df_silver.select(
        "produto",
        "categoria",
        "marca"
    )
    .distinct()
    .withColumn(
        "id",
        F.row_number().over(window_esp)   # ID sequencial começando em 1
    )
    .select("id", "produto", "categoria", "marca")
)

# salvar na Gold
(
    dim_produto.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(gold_path_produto)
)


26/03/12 01:11:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 01:11:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 01:11:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 01:11:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 01:11:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 01:11:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 0

In [106]:

# janela de ordenação para criar surrogate key
window_spec = Window.orderBy("pais")

dim_regiao = (
    df_silver
        .select(
            F.col("Pais").alias("pais"),
            "continente"
        )
        .distinct()
        .withColumn("id", F.row_number().over(window_spec))
        .select("id", "pais", "continente")
)

# salvar dimensão região (Gold)
(
    dim_regiao.write
        .format("delta")
        .mode("overwrite")
        .save(gold_path_regiao)
)


26/03/12 01:11:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 01:11:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 01:11:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 01:11:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 01:11:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 01:11:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 0

In [107]:
dim_regiao.show(truncate=False)

26/03/12 01:11:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 01:11:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 01:11:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---+--------------+----------------+
|id |pais          |continente      |
+---+--------------+----------------+
|1  |Alemanha      |Europa          |
|2  |Armênia       |Ásia            |
|3  |Austrália     |Austrália       |
|4  |Brasil        |América do Sul  |
|5  |Butão         |Ásia            |
|6  |Canadá        |América do Norte|
|7  |Chile         |América do Sul  |
|8  |China         |Ásia            |
|9  |Coreia do Sul |Ásia            |
|10 |Dinamarca     |Europa          |
|11 |Eslovênia     |Europa          |
|12 |Espanha       |Europa          |
|13 |Estados Unidos|América do Norte|
|14 |França        |Europa          |
|15 |Grécia        |Europa          |
|16 |Holanda       |Europa          |
|17 |Irlanda       |Europa          |
|18 |Irã           |Ásia            |
|19 |Itália        |Europa          |
|20 |Japão         |Ásia            |
+---+--------------+----------------+
only showing top 20 rows



26/03/12 01:11:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 01:11:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 01:11:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [108]:
dim_tempo = (
    df_silver
        .select("data_venda")
        .distinct()

        # ID no formato YYYYMMDD
        .withColumn(
            "id",
            F.date_format("data_venda", "yyyyMMdd").cast("int")
        )

        .withColumn("ano", F.year("data_venda"))
        .withColumn("mes", F.month("data_venda"))
        .withColumn("dia", F.dayofmonth("data_venda"))
        .withColumn("trimestre", F.quarter("data_venda"))
        .withColumn("dia_semana", F.dayofweek("data_venda"))
        .withColumn("nome_mes", F.date_format("data_venda", "MMMM"))

        .select(
            "id",
            "data_venda",
            "ano",
            "mes",
            "dia",
            "trimestre",
            "dia_semana",
            "nome_mes"
        )
)


(
    dim_tempo.write
        .format("delta")
        .mode("overwrite")
        .save(gold_path_tempo)
)


In [109]:
dim_tempo.show()

+--------+----------+----+---+---+---------+----------+---------+
|      id|data_venda| ano|mes|dia|trimestre|dia_semana| nome_mes|
+--------+----------+----+---+---+---------+----------+---------+
|20180528|2018-05-28|2018|  5| 28|        2|         2|      May|
|20180810|2018-08-10|2018|  8| 10|        3|         6|   August|
|20180317|2018-03-17|2018|  3| 17|        1|         7|    March|
|20180606|2018-06-06|2018|  6|  6|        2|         4|     June|
|20180626|2018-06-26|2018|  6| 26|        2|         3|     June|
|20180808|2018-08-08|2018|  8|  8|        3|         4|   August|
|20180811|2018-08-11|2018|  8| 11|        3|         7|   August|
|20180901|2018-09-01|2018|  9|  1|        3|         7|September|
|20180909|2018-09-09|2018|  9|  9|        3|         1|September|
|20180630|2018-06-30|2018|  6| 30|        2|         7|     June|
|20180323|2018-03-23|2018|  3| 23|        1|         6|    March|
|20180526|2018-05-26|2018|  5| 26|        2|         7|      May|
|20180908|

In [110]:
dim_cliente = spark.read.format("delta").load(gold_path_cliente)
dim_produto = spark.read.format("delta").load(gold_path_produto)
dim_regiao  = spark.read.format("delta").load(gold_path_regiao)
dim_tempo   = spark.read.format("delta").load(gold_path_tempo)


In [113]:
df_silver.count()

203882

In [114]:

# 1️⃣ Join com DimCliente
fato_vendas = df_silver.join(
    dim_cliente,
    df_silver.cliente_nome == dim_cliente.nome_completo,
    how="left"
).withColumnRenamed("id", "cliente_id")

# 2️⃣ Join com DimProduto
fato_vendas = fato_vendas.join(
    dim_produto,
    (fato_vendas.produto == dim_produto.produto) &
    (fato_vendas.categoria == dim_produto.categoria) &
    (fato_vendas.marca == dim_produto.marca),
    how="left"
).withColumnRenamed("id", "produto_id")

# 3️⃣ Join com DimRegiao
fato_vendas = fato_vendas.join(
    dim_regiao,
    fato_vendas.Pais == dim_regiao.pais,
    how="left"
).withColumnRenamed("id", "regiao_id")

# 4️⃣ Join com DimTempo
fato_vendas = fato_vendas.join(
    dim_tempo,
    fato_vendas.data_venda == dim_tempo.data_venda,
    how="left"
).withColumnRenamed("id", "tempo_id")

# 5️⃣ Selecionar colunas finais da tabela fato
fato_vendas = fato_vendas.select(
    "cliente_id",
    "produto_id",
    "regiao_id",
    "tempo_id",
    "quantidade",
    "preco_unitario",
    "total_venda",
)


# Salvar fato_vendas no MinIO (Delta) sem particionamento
fato_vendas.write.format("delta") \
    .mode("overwrite") \
    .save(gold_path_fato)

print("✅ fato_vendas salva no MinIO na camada Gold!")


✅ fato_vendas salva no MinIO na camada Gold!


In [115]:
fato_vendas.count()

203882

In [116]:
df_silver.count()

203882

In [ ]:
# Criar schema gold (se não existir)
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

DataFrame[]

In [ ]:
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|  default|
|     gold|
+---------+



In [ ]:

# Dim Cliente
spark.sql("""
    CREATE TABLE IF NOT EXISTS gold.dim_vendas
    USING DELTA
    LOCATION 's3a://analytics/medallion/03-gold/dim_cliente'
""")

DataFrame[]

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS gold.dim_produto
    USING DELTA
    LOCATION 's3a://analytics/medallion/03-gold/dim_produto'
""")

DataFrame[]

In [ ]:

# Dim Regiao
spark.sql("""
    CREATE TABLE IF NOT EXISTS gold.dim_regiao
    USING DELTA
    LOCATION 's3a://analytics/medallion/03-gold/dim_regiao'
""")

DataFrame[]

In [ ]:

# Dim Tempo
spark.sql("""
    CREATE TABLE IF NOT EXISTS gold.dim_tempo
    USING DELTA
    LOCATION 's3a://analytics/medallion/03-gold/dim_tempo'
""")


DataFrame[]

In [ ]:
spark.sql("SHOW TABLES IN gold").show()

+---------+-----------+-----------+
|namespace|  tableName|isTemporary|
+---------+-----------+-----------+
|     gold|dim_produto|      false|
|     gold| dim_regiao|      false|
|     gold|  dim_tempo|      false|
|     gold| dim_vendas|      false|
+---------+-----------+-----------+



In [117]:
spark.stop()